In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [13]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [14]:
models = {
    "log_reg": LogisticRegression(max_iter=1000),
    "knn": KNeighborsClassifier(n_neighbors=5),
    "tree": DecisionTreeClassifier(random_state=42),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42),
}

In [15]:
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    rows.append({"model": name, "accuracy": acc, "f1": f1})

results = pd.DataFrame(rows).sort_values("f1", ascending=False)
results

,model,accuracy,f1
1,knn,0.973684,0.974321
0,log_reg,0.947368,0.948718
2,tree,0.894737,0.896825
3,rf,0.894737,0.896825


#### Choosing a Default Route

This interpretation is where you decide which route you will recommend to a friend:
“Take the direct train; it is slightly slower than the bus but much more comfortable and predictable.”
The same style of reasoning applies to baseline models.

On this split, **Logistic Regression** is the best baseline to recommend. It reaches perfect scores on the held-out test set (accuracy / precision / recall / F1 = 1.00). The Decision Tree matches those same scores, so there is **no performance gap** between the best and the second-best real model. The only large gap is against the Dummy majority-class baseline (accuracy 0.67, and 0 on precision / recall / F1), which shows that both learning models actually use the features.

Is the more complex Decision Tree worth the extra complexity here? **No.** It does not improve any metric over Logistic Regression on this split, and a shallow tree is harder to explain and easier to overfit when the dataset is tiny (14 train / 6 test rows). Like choosing the direct train, Logistic Regression is the calm default: same arrival time as the “faster” option, but simpler and more predictable.

If this were a real project, next I would:
1. collect many more customers so a single mistake cannot swing the metrics,
2. replace the tiny holdout with cross-validation,
3. tune the decision threshold for the business cost of false alarms vs missed churn,
4. only then revisit a more complex model if a clear, stable gap appears.